# Comparing Various Byte Pair Encoding (BPE) Implementations

## 1. Using BPE from `tiktoken`

In [1]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [2]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [3]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [4]:
print(tik_tokenizer.n_vocab)

50257


## 2. Using the original BPE implementation used in GPT-2

In [5]:
from bpe_openai_gpt2 import get_encoder, download_vocab

In [6]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 1.21Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 790kit/s]                                                        


In [7]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [8]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [9]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


## 3. Using the BPE via Hugging Face transformers

In [10]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [12]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [13]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

## 4. Using my own from-scratch BPE tokenizer

In [14]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [15]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [16]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


## 5. A quick performance benchmark

In [17]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

### 5.1 Original OpenAI GPT-2 tokenizer

In [18]:
%timeit orig_tokenizer.encode(raw_text)

4.98 ms ± 143 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### 5.2 Tiktoken OpenAI GPT-2 tokenizer

In [19]:
%timeit tik_tokenizer.encode(raw_text)

1.58 ms ± 59.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### 5.3 Hugging Face OpenAI GPT-2 tokenizer

In [20]:
%timeit hf_tokenizer(raw_text)["input_ids"]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


6.62 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

6.59 ms ± 168 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

6.6 ms ± 68.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

### 5.4 My own GPT-2 tokenizer (for educational purposes)

In [24]:
%timeit tokenizer_gpt2.encode(raw_text)

13.9 ms ± 226 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
